In [ ]:
# Model 1: no specific constraint
# -------------- better display --------------
from gurobipy import Model, GRB, quicksum
import pandas as pd
import scipy.stats as stats
from scipy.special import gamma, gammaincc
import numpy as np
import math

# -------------- data --------------
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv', sep=',')
N = 224
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
T_years = 17
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]
Age_2023 = new_df['Age 2023'].head(N).tolist()

gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584

C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER = 1.6
mu_MANT  = 4.2
M_alpha = max(alpha_init) * 10

first_year_leaks = [Y * alpha_init[i] * K[i] for i in range(N)]
# print("第一年泄漏量(kg):", [f"{x:.1f}" for x in first_year_leaks])

# def compute_gamma_h(age, gamma_alpha, gamma_beta):
#     z = gamma_beta * age
#     if z <= 0:
#         return 0.0
#     num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z) 
#     den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)  # Γ(α, z)
#     return num / den


def compute_gamma_h(age, gamma_alpha, gamma_beta):
    upper_gamma = gammaincc(gamma_alpha, gamma_beta * age) * gamma(gamma_alpha)
    
    h_i_age = (gamma_beta**gamma_alpha * age**(gamma_alpha - 1) * 
               np.exp(-gamma_beta * age)) / upper_gamma
    return h_i_age



mdl = Model("leak_decision_final")
mdl.Params.OutputFlag = 1

# -------------- decision variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, name="alpha")
last_action = mdl.addVars(N, T_years+1, vtype=GRB.BINARY, ub=T_years, name="last_action")
#  for leak tracking
cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")

# initialization
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i]/compute_gamma_h(Age_2023[i], gamma_alpha, gamma_beta) * compute_gamma_h(Age_2023[i]+1, gamma_alpha, gamma_beta))
    mdl.addConstr(last_action[i,0] == 0)

# alpha evolution
for i in range(N):
    for t in range(1, T_years+1):
        # no action: growth follows gamma distribution
        growth = (alpha[i,0]/compute_gamma_h(Age_2023[i], gamma_alpha, gamma_beta)) * compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta)
        no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(alpha[i,t] >= growth - M_alpha * (1 - no_action))
        mdl.addConstr(alpha[i,t] <= growth + M_alpha * (1 - no_action))
        
        # COPER maintenance: precise to alpha[t-1]/1.6
        coper_target = alpha[i,t-1] / mu_COPER
        mdl.addConstr(alpha[i,t] <= coper_target + M_alpha * (1 - x_COPER[i,t-1]))
        mdl.addConstr(alpha[i,t] >= coper_target - M_alpha * (1 - x_COPER[i,t-1]))
        
        # MANT maintenance: precise to alpha[t-1]/4.2
        mant_target = alpha[i,t-1] / mu_MANT
        mdl.addConstr(alpha[i,t] <= mant_target + M_alpha * (1 - x_MANT[i,t-1]))
        mdl.addConstr(alpha[i,t] >= mant_target - M_alpha * (1 - x_MANT[i,t-1]))

# last_action update
M_time = T_years + 5
for i in range(N):
    for t in range(1, T_years+1):
        action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
        # if action, last_action = t-1
        mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
        mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
        # if no action, last_action inherits
        mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
        mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

# yearly leak calculation
BIG_M_LEAK = 1e6
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(yearly_leak[i,t] == (Y * alpha[i,t] * (10**5)) * K[i])

# cumulative leak logic
LEAK_TH = 2.0
for i in range(N):
    # year 0: cum_leak = yearly_leak
    mdl.addConstr(cum_leak[i,0] == yearly_leak[i,0])
    
    # leak accumulation logic:
    for t in range(1, T_years):
        # case 1: no action last year → cum_leak[t] = cum_leak[t-1] + yearly_leak[t]
        no_action_prev = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(cum_leak[i,t] >= cum_leak[i,t-1] + yearly_leak[i,t] - BIG_M_LEAK * (1 - no_action_prev))
        mdl.addConstr(cum_leak[i,t] <= cum_leak[i,t-1] + yearly_leak[i,t] + BIG_M_LEAK * (1 - no_action_prev))
        
        # case 2: last year had action → cum_leak[t] = yearly_leak[t]（reset）
        action_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        mdl.addConstr(cum_leak[i,t] >= yearly_leak[i,t] - BIG_M_LEAK * (1 - action_prev))
        mdl.addConstr(cum_leak[i,t] <= yearly_leak[i,t] + BIG_M_LEAK * (1 - action_prev))
    
    # leak trigger constraint: only if cum_leak >= 2kg can we have another action 
    for t in range(T_years):
        x_this_year = x_COPER[i,t] + x_MANT[i,t]
        mdl.addConstr(x_this_year <= 1)  # at most one action per year
        mdl.addConstr(cum_leak[i,t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year)) #！！！！！！

# objective
leak_cost = quicksum(
    (Y * alpha[i,t] * (10**5)) * K[i] * C_leak_t[t]
    for i in range(N) for t in range(T_years)
)
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# solve parameters
mdl.Params.MIPGap = 0.05
mdl.Params.TimeLimit = 300
mdl.Params.IntFeasTol = 1e-6
mdl.Params.NumericFocus = 2

mdl.optimize()

# output results
if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ Total Cost: {mdl.objVal:,.0f}")
    
    print("\n=== Detailed Maintenance Plan and Cumulative Leak Verification ===")
    total_leak_cost = 0
    total_action_cost = 0
    
    for i in range(N):
        # act_type = "Init"   # 或者 "No action"
        print(f"\n{'='*70}")
        print(f"CSEM {i+1} (Initial α={alpha_init[i]:.6f}, Volume={V[i]:.1f}m³, csemCur = {new_df['csemCur'].iloc[i]:.1f}), K={K[i]:.6f}, ΔP={Delta_P[i]:.3f}bar, Age2023={Age_2023[i]:.1f} years")
        print(f"{'='*70}")
        
        csem_leak_cost = 0
        csem_action_cost = 0
        
        for t in range(T_years):
            if t == 0:
                alpha_start = alpha_init[i]
                alpha_end = alpha[i,t].X
                leak_kg = yearly_leak[i,t].X
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
            else:
                alpha_start = alpha[i,t-1].X
                alpha_end = alpha[i,t].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
                
                act_type = ""
                act_cost = 0
                if x_COPER[i,t-1].X > 0.5:
                    act_type = "COPER"
                    act_cost = C_COPER
                    csem_action_cost += act_cost
                elif x_MANT[i,t-1].X > 0.5:
                    act_type = "MANT" 
                    act_cost = C_MANT
                    csem_action_cost += act_cost
                else:
                    act_type = "No action"
                
                total_leak_cost += leak_cost_t
                if act_cost > 0:
                    total_action_cost += act_cost
                
            alpha_reduction = (alpha_start - alpha_end) / alpha_start * 100 if alpha_start > 0 else 0
            cum_leak_val = cum_leak[i,t].X
            valid_check = "❌" if (cum_leak_val >= 2) else "✅"
            
            print(f"  Y{t+1:2d}: {act_type:<8} | "
                  f"α: {alpha_start:.6f}→{alpha_end:.6f} "
                  f"(-{alpha_reduction:5.1f}%) | "
                  f"Leak{leak_kg:>6.3f}kg | "
                  f"Cum_Leak{cum_leak_val:>6.3f}kg | {valid_check} | " 
                  f"CR {365 * alpha_end / Delta_P[i]:.3f}") # new add CR display
        
        print(f"  CSEM{i+1} Action cost{csem_action_cost:>9,.0f} + Leak cost{csem_leak_cost:>10,.0f} = Total {csem_action_cost+csem_leak_cost:>11,.0f}")
        print(f"  Total leak: {sum(yearly_leak[i,t].X for t in range(T_years)):.1f} kg")
        print(f"  Total COPER action: {sum(x_COPER[i,t].X for t in range(T_years)):.0f}")
        print(f"  Total MANT action: {sum(x_MANT[i,t].X for t in range(T_years)):.0f}")
    
    print(f"\n{'='*70}")
    print(f"🎯 Total: action cost {total_action_cost:>12,.0f} + leak cost {total_leak_cost:>12,.0f} = total cost {mdl.objVal:>12,.0f}")
    print(f"{'='*70}")
    print(f"Total leak over entire horizon: {sum(yearly_leak[i,t].X for i in range(N) for t in range(T_years)):.1f} kg")
    print(f"Total COPER actions: {sum(x_COPER[i,t].X for i in range(N) for t in range(T_years)):.0f}")
    print(f"Total MANT actions: {sum(x_MANT[i,t].X for i in range(N) for t in range(T_years)):.0f}")

    print('h value 2023 for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {compute_gamma_h(Age_2023[i], gamma_alpha, gamma_beta):.6f}")
    print('h values for each CSEM over years:') 
    for i in range(N):
        print(f"  CSEM {i+1}: {[h_cr for h_cr in [compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta) for t in range(T_years)]]}")
    print('h values values for 2023:')
    for i in range(N):
        print(f"  CSEM {i+1}: {compute_gamma_h(Age_2023[i], gamma_alpha, gamma_beta):.6f}")

    print('alpha values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: { [alpha[i,t].X for t in range(T_years)] }")
    print('alpha 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {alpha_init[i]:.6f}")
    # print ('Initial number of CR for each CSEM:')
    # for i in range(N):
    #     print(f"  CSEM {i+1}: {365 * alpha_init[i]/Delta_P[i]:.0f}, csemCur: {new_df['csemCur'].iloc[i]:.1f}")

    # 计算初始 CR 并转为整数（处理 Delta_P=0）
    initial_cr_float = np.array([365 * alpha_init[i] / Delta_P[i] if Delta_P[i] > 1e-6 else 0 
                                for i in range(N)])
    initial_cr = np.round(initial_cr_float).astype(int)  # 四舍五入到整数

    non_zero_count = np.count_nonzero(initial_cr)
    print(f" number of non-zero CR: {non_zero_count}/{N} ({non_zero_count/N*100:.1f}%)")

    for i in range(N):
        cr_int = int(round(365 * alpha_init[i] / Delta_P[i])) if Delta_P[i] > 1e-6 else 0
        print(f"  CSEM {i+1}: {cr_int}, csemCur: {new_df['csemCur'].iloc[i]:.1f}")
    
else:
    print(f"❌ Optimization status: {mdl.status}")


TypeError: return arrays must be of ArrayType

In [ ]:
# Model 2: budget constraint
# -------------- better display --------------
from gurobipy import Model, GRB, quicksum
import pandas as pd
import scipy.stats as stats
from scipy.special import gamma, gammaincc
import numpy as np

# -------------- data --------------
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv', sep=',')
N = 10
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
T_years = 17
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]
Age_2023 = new_df['Age 2023'].head(N).tolist()
Total_budget = 27000     # total annual budget

gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584

C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER = 1.6
mu_MANT  = 4.2
M_alpha = max(alpha_init) * 10

first_year_leaks = [Y * alpha_init[i] * K[i] for i in range(N)]
# print("第一年泄漏量(kg):", [f"{x:.1f}" for x in first_year_leaks])

def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z) 
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)  # Γ(α, z)
    return num / den


mdl = Model("leak_decision_final")
mdl.Params.OutputFlag = 1

# -------------- decision variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, name="alpha")
last_action = mdl.addVars(N, T_years+1, vtype=GRB.BINARY, ub=T_years, name="last_action")
#  for leak tracking
cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")

# initialization
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i]/compute_gamma_h(Age_2023[i], gamma_alpha, gamma_beta) * compute_gamma_h(Age_2023[i]+1, gamma_alpha, gamma_beta))
    mdl.addConstr(last_action[i,0] == 0)

# alpha evolution
for i in range(N):
    for t in range(1, T_years+1):
        # no action: growth follows gamma distribution
        growth = (alpha[i,0]/compute_gamma_h(Age_2023[i], gamma_alpha, gamma_beta)) * compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta)
        no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(alpha[i,t] >= growth - M_alpha * (1 - no_action))
        mdl.addConstr(alpha[i,t] <= growth + M_alpha * (1 - no_action))
        
        # COPER maintenance: precise to alpha[t-1]/1.6
        coper_target = alpha[i,t-1] / mu_COPER
        mdl.addConstr(alpha[i,t] <= coper_target + M_alpha * (1 - x_COPER[i,t-1]))
        mdl.addConstr(alpha[i,t] >= coper_target - M_alpha * (1 - x_COPER[i,t-1]))
        
        # MANT maintenance: precise to alpha[t-1]/4.2
        mant_target = alpha[i,t-1] / mu_MANT
        mdl.addConstr(alpha[i,t] <= mant_target + M_alpha * (1 - x_MANT[i,t-1]))
        mdl.addConstr(alpha[i,t] >= mant_target - M_alpha * (1 - x_MANT[i,t-1]))

# last_action update
M_time = T_years + 5
for i in range(N):
    for t in range(1, T_years+1):
        action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
        # if action, last_action = t-1
        mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
        mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
        # if no action, last_action inherits
        mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
        mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

# yearly leak calculation
BIG_M_LEAK = 1e6
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(yearly_leak[i,t] == (Y * alpha[i,t] * (10**5)) * K[i])

# cumulative leak logic
LEAK_TH = 2.0
for i in range(N):
    # year 0: cum_leak = yearly_leak
    mdl.addConstr(cum_leak[i,0] == yearly_leak[i,0])
    
    # leak accumulation logic:
    for t in range(1, T_years):
        # case 1: no action last year → cum_leak[t] = cum_leak[t-1] + yearly_leak[t]
        no_action_prev = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(cum_leak[i,t] >= cum_leak[i,t-1] + yearly_leak[i,t] - BIG_M_LEAK * (1 - no_action_prev))
        mdl.addConstr(cum_leak[i,t] <= cum_leak[i,t-1] + yearly_leak[i,t] + BIG_M_LEAK * (1 - no_action_prev))
        
        # case 2: last year had action → cum_leak[t] = yearly_leak[t]（reset）
        action_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        mdl.addConstr(cum_leak[i,t] >= yearly_leak[i,t] - BIG_M_LEAK * (1 - action_prev))
        mdl.addConstr(cum_leak[i,t] <= yearly_leak[i,t] + BIG_M_LEAK * (1 - action_prev))
    
    # leak trigger constraint: only if cum_leak >= 2kg can we have another action 
    for t in range(T_years):
        x_this_year = x_COPER[i,t] + x_MANT[i,t]
        mdl.addConstr(x_this_year <= 1)  # at most one action per year
        mdl.addConstr(cum_leak[i,t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year)) #！！！！！！

# Annual action budget constraint
for t in range(T_years):
    yearly_action_cost = quicksum(C_COPER * x_COPER[i, t] + C_MANT * x_MANT[i, t] for i in range(N))
    mdl.addConstr(yearly_action_cost <= Total_budget, name=f"YearlyBudgetLimit[{t}]")


# objective
leak_cost = quicksum(
    (Y * alpha[i,t] * (10**5)) * K[i] * C_leak_t[t]
    for i in range(N) for t in range(T_years)
)
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# solve parameters
mdl.Params.MIPGap = 0.05
mdl.Params.TimeLimit = 300
mdl.Params.IntFeasTol = 1e-6
mdl.Params.NumericFocus = 2

mdl.optimize()

# output results
if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ Total Cost: {mdl.objVal:,.0f}")
    
    print("\n=== Detailed Maintenance Plan and Cumulative Leak Verification ===")
    total_leak_cost = 0
    total_action_cost = 0
    
    for i in range(N):
        print(f"\n{'='*70}")
        print(f"CSEM {i+1} (Initial α={alpha_init[i]:.6f}, Volume={V[i]:.1f}m³, csemCur = {new_df['csemCur'].iloc[i]:.1f})")
        print(f"{'='*70}")
        
        csem_leak_cost = 0
        csem_action_cost = 0
        
        for t in range(T_years):
            if t == 0:
                alpha_start = alpha_init[i]
                alpha_end = alpha[i,t].X
                leak_kg = yearly_leak[i,t].X
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
            else:
                alpha_start = alpha[i,t-1].X
                alpha_end = alpha[i,t].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
                
                act_type = ""
                act_cost = 0
                if x_COPER[i,t].X > 0.5:
                    act_type = "COPER"
                    act_cost = C_COPER
                    csem_action_cost += act_cost
                elif x_MANT[i,t].X > 0.5:
                    act_type = "MANT" 
                    act_cost = C_MANT
                    csem_action_cost += act_cost
                else:
                    act_type = "No action"
                
                total_leak_cost += leak_cost_t
                if act_cost > 0:
                    total_action_cost += act_cost
                
            alpha_reduction = (alpha_start - alpha_end) / alpha_start * 100 if alpha_start > 0 else 0
            cum_leak_val = cum_leak[i,t].X
            valid_check = "❌" if (cum_leak_val >= 2) else "✅"
            
            print(f"  Y{t+1:2d}: {act_type:<8} | "
                  f"α: {alpha_start:.6f}→{alpha_end:.6f} "
                  f"(-{alpha_reduction:5.1f}%) | "
                  f"Leak{leak_kg:>6.3f}kg | "
                  f"Cum_Leak{cum_leak_val:>6.3f}kg | {valid_check}")
        
        print(f"  CSEM{i+1} Action cost{csem_action_cost:>9,.0f} + Leak cost{csem_leak_cost:>10,.0f} = Total {csem_action_cost+csem_leak_cost:>11,.0f}")
        print(f"  Total leak: {sum(yearly_leak[i,t].X for t in range(T_years)):.1f} kg")
        print(f"  Total COPER action: {sum(x_COPER[i,t].X for t in range(T_years)):.0f}")
        print(f"  Total MANT action: {sum(x_MANT[i,t].X for t in range(T_years)):.0f}")
    
    print(f"\n{'='*70}")
    print(f"🎯 Total: action cost {total_action_cost:>12,.0f} + leak cost {total_leak_cost:>12,.0f} = total cost {mdl.objVal:>12,.0f}")
    print(f"{'='*70}")

    print('h values for each CSEM over years:')
    for i in range(N):
        print(f"  CSEM {i+1}: {[h_cr for h_cr in [compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta) for t in range(T_years)]]}")
    print('alpha values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: { [alpha[i,t].X for t in range(T_years)] }")
    print('alpha 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {alpha_init[i]:.6f}")

    
else:
    print(f"❌ Optimization status: {mdl.status}")


Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 300
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 2897 rows, 1040 columns and 10230 nonzeros
Model fingerprint: 0x09e79dcd
Variable types: 520 continuous, 520 integer (520 binary)
Coefficient statistics:
  Matrix range     [9e-03, 1e+06]
  Objective range  [2e+04, 9e+07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-04, 1e+06]
Presolve removed 2842 rows and 1013 columns
Presolve time: 0.00s
Presolved: 55 rows, 27 columns, 171 nonzeros
Variable types: 13 continuous, 14 integer (14 binary)
Found heuristic solution: objective 3934161.8787

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.01 work units)
Thread count was 12 (of 12 available processors)

Solution count 1: 3.93416e+06 

O

In [27]:
# Model 3: resource constraint
# -------------- better display --------------
from gurobipy import Model, GRB, quicksum
import pandas as pd
import scipy.stats as stats
from scipy.special import gamma, gammaincc
import numpy as np

# -------------- data --------------
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv', sep=',')
N = 10
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
T_years = 17
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]
Age_2023 = new_df['Age 2023'].head(N).tolist()
L_total = 1        # total maintenance actions

gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584

C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER = 1.6
mu_MANT  = 4.2
M_alpha = max(alpha_init) * 10

first_year_leaks = [Y * alpha_init[i] * K[i] for i in range(N)]
# print("第一年泄漏量(kg):", [f"{x:.1f}" for x in first_year_leaks])

def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z) 
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)  # Γ(α, z)
    return num / den


mdl = Model("leak_decision_final")
mdl.Params.OutputFlag = 1

# -------------- decision variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, name="alpha")
last_action = mdl.addVars(N, T_years+1, vtype=GRB.BINARY, ub=T_years, name="last_action")
#  for leak tracking
cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")

# initialization
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i]/compute_gamma_h(Age_2023[i], gamma_alpha, gamma_beta) * compute_gamma_h(Age_2023[i]+1, gamma_alpha, gamma_beta))
    mdl.addConstr(last_action[i,0] == 0)

# alpha evolution
for i in range(N):
    for t in range(1, T_years+1):
        # no action: growth follows gamma distribution
        growth = (alpha[i,0]/compute_gamma_h(Age_2023[i], gamma_alpha, gamma_beta)) * compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta)
        no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(alpha[i,t] >= growth - M_alpha * (1 - no_action))
        mdl.addConstr(alpha[i,t] <= growth + M_alpha * (1 - no_action))
        
        # COPER maintenance: precise to alpha[t-1]/1.6
        coper_target = alpha[i,t-1] / mu_COPER
        mdl.addConstr(alpha[i,t] <= coper_target + M_alpha * (1 - x_COPER[i,t-1]))
        mdl.addConstr(alpha[i,t] >= coper_target - M_alpha * (1 - x_COPER[i,t-1]))
        
        # MANT maintenance: precise to alpha[t-1]/4.2
        mant_target = alpha[i,t-1] / mu_MANT
        mdl.addConstr(alpha[i,t] <= mant_target + M_alpha * (1 - x_MANT[i,t-1]))
        mdl.addConstr(alpha[i,t] >= mant_target - M_alpha * (1 - x_MANT[i,t-1]))

# last_action update
M_time = T_years + 5
for i in range(N):
    for t in range(1, T_years+1):
        action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
        # if action, last_action = t-1
        mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
        mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
        # if no action, last_action inherits
        mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
        mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

# yearly leak calculation
BIG_M_LEAK = 1e6
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(yearly_leak[i,t] == (Y * alpha[i,t] * (10**5)) * K[i])

# cumulative leak logic
LEAK_TH = 2.0
for i in range(N):
    # year 0: cum_leak = yearly_leak
    mdl.addConstr(cum_leak[i,0] == yearly_leak[i,0])
    
    # leak accumulation logic:
    for t in range(1, T_years):
        # case 1: no action last year → cum_leak[t] = cum_leak[t-1] + yearly_leak[t]
        no_action_prev = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(cum_leak[i,t] >= cum_leak[i,t-1] + yearly_leak[i,t] - BIG_M_LEAK * (1 - no_action_prev))
        mdl.addConstr(cum_leak[i,t] <= cum_leak[i,t-1] + yearly_leak[i,t] + BIG_M_LEAK * (1 - no_action_prev))
        
        # case 2: last year had action → cum_leak[t] = yearly_leak[t]（reset）
        action_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        mdl.addConstr(cum_leak[i,t] >= yearly_leak[i,t] - BIG_M_LEAK * (1 - action_prev))
        mdl.addConstr(cum_leak[i,t] <= yearly_leak[i,t] + BIG_M_LEAK * (1 - action_prev))
    
    # leak trigger constraint: only if cum_leak >= 2kg can we have another action 
    for t in range(T_years):
        x_this_year = x_COPER[i,t] + x_MANT[i,t]
        mdl.addConstr(x_this_year <= 1)  # at most one action per year
        mdl.addConstr(cum_leak[i,t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year)) #！！！！！！

# Yearly tasks limit
for t in range(T_years):
    mdl.addConstr(quicksum(x_COPER[i,t] + x_MANT[i,t] for i in range(N)) <= L_total, name=f"Yearly_limit_t{t}")

# objective
leak_cost = quicksum(
    (Y * alpha[i,t] * (10**5)) * K[i] * C_leak_t[t]
    for i in range(N) for t in range(T_years)
)
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# solve parameters
mdl.Params.MIPGap = 0.05
mdl.Params.TimeLimit = 300
mdl.Params.IntFeasTol = 1e-6
mdl.Params.NumericFocus = 2

mdl.optimize()

# output results
if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ Total Cost: {mdl.objVal:,.0f}")
    
    print("\n=== Detailed Maintenance Plan and Cumulative Leak Verification ===")
    total_leak_cost = 0
    total_action_cost = 0
    
    for i in range(N):
        print(f"\n{'='*70}")
        print(f"CSEM {i+1} (Initial α={alpha_init[i]:.6f}, Volume={V[i]:.1f}m³, csemCur = {new_df['csemCur'].iloc[i]:.1f})")
        print(f"{'='*70}")
        
        csem_leak_cost = 0
        csem_action_cost = 0
        
        for t in range(T_years):
            if t == 0:
                alpha_start = alpha_init[i]
                alpha_end = alpha[i,t].X
                leak_kg = yearly_leak[i,t].X
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
            else:
                alpha_start = alpha[i,t-1].X
                alpha_end = alpha[i,t].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
                
                act_type = ""
                act_cost = 0
                if x_COPER[i,t].X > 0.5:
                    act_type = "COPER"
                    act_cost = C_COPER
                    csem_action_cost += act_cost
                elif x_MANT[i,t].X > 0.5:
                    act_type = "MANT" 
                    act_cost = C_MANT
                    csem_action_cost += act_cost
                else:
                    act_type = "No action"
                
                total_leak_cost += leak_cost_t
                if act_cost > 0:
                    total_action_cost += act_cost
                
            alpha_reduction = (alpha_start - alpha_end) / alpha_start * 100 if alpha_start > 0 else 0
            cum_leak_val = cum_leak[i,t].X
            valid_check = "❌" if (cum_leak_val >= 2) else "✅"
            
            print(f"  Y{t+1:2d}: {act_type:<8} | "
                  f"α: {alpha_start:.6f}→{alpha_end:.6f} "
                  f"(-{alpha_reduction:5.1f}%) | "
                  f"Leak{leak_kg:>6.3f}kg | "
                  f"Cum_Leak{cum_leak_val:>6.3f}kg | {valid_check}")
        
        print(f"  CSEM{i+1} Action cost{csem_action_cost:>9,.0f} + Leak cost{csem_leak_cost:>10,.0f} = Total {csem_action_cost+csem_leak_cost:>11,.0f}")
        print(f"  Total leak: {sum(yearly_leak[i,t].X for t in range(T_years)):.1f} kg")
        print(f"  Total COPER action: {sum(x_COPER[i,t].X for t in range(T_years)):.0f}")
        print(f"  Total MANT action: {sum(x_MANT[i,t].X for t in range(T_years)):.0f}")
    
    print(f"\n{'='*70}")
    print(f"🎯 Total: action cost {total_action_cost:>12,.0f} + leak cost {total_leak_cost:>12,.0f} = total cost {mdl.objVal:>12,.0f}")
    print(f"{'='*70}")

    print('h values for each CSEM over years:')
    for i in range(N):
        print(f"  CSEM {i+1}: {[h_cr for h_cr in [compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta) for t in range(T_years)]]}")
    print('alpha values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: { [alpha[i,t].X for t in range(T_years)] }")
    print('alpha 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {alpha_init[i]:.6f}")


    
else:
    print(f"❌ Optimization status: {mdl.status}")


Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 300
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 2897 rows, 1040 columns and 10230 nonzeros
Model fingerprint: 0xab9d00bf
Variable types: 520 continuous, 520 integer (520 binary)
Coefficient statistics:
  Matrix range     [9e-03, 1e+06]
  Objective range  [2e+04, 9e+07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-04, 1e+06]
Presolve removed 2782 rows and 972 columns
Presolve time: 0.01s
Presolved: 115 rows, 68 columns, 603 nonzeros
Variable types: 32 continuous, 36 integer (36 binary)
Found heuristic solution: objective 3937736.2040

Root relaxation: objective 3.907090e+06, 23 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     W

In [33]:
# Model 4: budget + resource constraint
# -------------- better display --------------
from gurobipy import Model, GRB, quicksum
import pandas as pd
import scipy.stats as stats
from scipy.special import gamma, gammaincc
import numpy as np

# -------------- data --------------
new_df = pd.read_csv('/Users/wenbo.wu/Documents/Documents - Wenbo WU/Thesis/RTE_use_case/RTE_dashboard/RTE/new_df_AlphaCorrected.csv', sep=',')
N = 224
alpha_init = new_df['alpha'].head(N).tolist()
V = new_df['Volume'].head(N).tolist()
K = [146 * V[i] / (8.314 * 293.15 * (10 ** 3)) for i in range(N)]
T_years = 17
Y = 365
Delta_P = [round(float(new_df.iloc[i]['Pression Nominale Relative'] - new_df['CR_threshold'].iloc[i]), 3) 
           for i in range(N)]
Age_2023 = new_df['Age 2023'].head(N).tolist()
# Total_budget = 27000    # total annual budget
Total_budget = 604800
L_total = 16        # total maintenance actions

gamma_alpha = 5.682011911867158 
gamma_beta = 0.25018840842979584

C_COPER = 15600
C_MANT  = 48000
c_factors = [0.0952 + 0.004 * t for t in range(T_years)]
C_leak_t = [c * 25200 for c in c_factors]
mu_COPER = 1.6
mu_MANT  = 4.2
M_alpha = max(alpha_init) * 10

first_year_leaks = [Y * alpha_init[i] * K[i] for i in range(N)]
# print("第一年泄漏量(kg):", [f"{x:.1f}" for x in first_year_leaks])

def compute_gamma_h(age, gamma_alpha, gamma_beta):
    z = gamma_beta * age
    if z <= 0:
        return 0.0
    num = z ** (gamma_alpha - 1) * gamma_beta * np.exp(-z) 
    den = gamma(gamma_alpha) * gammaincc(gamma_alpha, z)  # Γ(α, z)
    return num / den


mdl = Model("leak_decision_final")
mdl.Params.OutputFlag = 1

# -------------- decision variables --------------
x_COPER = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_COPER")
x_MANT  = mdl.addVars(N, T_years, vtype=GRB.BINARY, name="x_MANT")
alpha = mdl.addVars(N, T_years+1, name="alpha")
last_action = mdl.addVars(N, T_years+1, vtype=GRB.BINARY, ub=T_years, name="last_action")
#  for leak tracking
cum_leak = mdl.addVars(N, T_years, lb=0, name="cum_leak")
yearly_leak = mdl.addVars(N, T_years, lb=0, name="yearly_leak")

# initialization
for i in range(N):
    mdl.addConstr(alpha[i,0] == alpha_init[i]/compute_gamma_h(Age_2023[i], gamma_alpha, gamma_beta) * compute_gamma_h(Age_2023[i]+1, gamma_alpha, gamma_beta))
    mdl.addConstr(last_action[i,0] == 0)

# alpha evolution
for i in range(N):
    for t in range(1, T_years+1):
        # no action: growth follows gamma distribution
        growth = (alpha[i,0]/compute_gamma_h(Age_2023[i], gamma_alpha, gamma_beta)) * compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta)
        no_action = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(alpha[i,t] >= growth - M_alpha * (1 - no_action))
        mdl.addConstr(alpha[i,t] <= growth + M_alpha * (1 - no_action))
        
        # COPER maintenance: precise to alpha[t-1]/1.6
        coper_target = alpha[i,t-1] / mu_COPER
        mdl.addConstr(alpha[i,t] <= coper_target + M_alpha * (1 - x_COPER[i,t-1]))
        mdl.addConstr(alpha[i,t] >= coper_target - M_alpha * (1 - x_COPER[i,t-1]))
        
        # MANT maintenance: precise to alpha[t-1]/4.2
        mant_target = alpha[i,t-1] / mu_MANT
        mdl.addConstr(alpha[i,t] <= mant_target + M_alpha * (1 - x_MANT[i,t-1]))
        mdl.addConstr(alpha[i,t] >= mant_target - M_alpha * (1 - x_MANT[i,t-1]))

# last_action update
M_time = T_years + 5
for i in range(N):
    for t in range(1, T_years+1):
        action_t1 = x_COPER[i,t-1] + x_MANT[i,t-1]
        # if action, last_action = t-1
        mdl.addConstr(last_action[i,t] >= (t-1) - M_time * (1 - action_t1))
        mdl.addConstr(last_action[i,t] <= (t-1) + M_time * (1 - action_t1))
        # if no action, last_action inherits
        mdl.addConstr(last_action[i,t] >= last_action[i,t-1] - M_time * action_t1)
        mdl.addConstr(last_action[i,t] <= last_action[i,t-1] + M_time * action_t1)

# yearly leak calculation
BIG_M_LEAK = 1e6
for i in range(N):
    for t in range(T_years):
        mdl.addConstr(yearly_leak[i,t] == (Y * alpha[i,t] * (10**5)) * K[i])

# cumulative leak logic
LEAK_TH = 2.0
for i in range(N):
    # year 0: cum_leak = yearly_leak
    mdl.addConstr(cum_leak[i,0] == yearly_leak[i,0])
    
    # leak accumulation logic:
    for t in range(1, T_years):
        # case 1: no action last year → cum_leak[t] = cum_leak[t-1] + yearly_leak[t]
        no_action_prev = 1 - (x_COPER[i,t-1] + x_MANT[i,t-1])
        mdl.addConstr(cum_leak[i,t] >= cum_leak[i,t-1] + yearly_leak[i,t] - BIG_M_LEAK * (1 - no_action_prev))
        mdl.addConstr(cum_leak[i,t] <= cum_leak[i,t-1] + yearly_leak[i,t] + BIG_M_LEAK * (1 - no_action_prev))
        
        # case 2: last year had action → cum_leak[t] = yearly_leak[t]（reset）
        action_prev = x_COPER[i,t-1] + x_MANT[i,t-1]
        mdl.addConstr(cum_leak[i,t] >= yearly_leak[i,t] - BIG_M_LEAK * (1 - action_prev))
        mdl.addConstr(cum_leak[i,t] <= yearly_leak[i,t] + BIG_M_LEAK * (1 - action_prev))
    
    # leak trigger constraint: only if cum_leak >= 2kg can we have another action 
    for t in range(T_years):
        x_this_year = x_COPER[i,t] + x_MANT[i,t]
        mdl.addConstr(x_this_year <= 1)  # at most one action per year
        mdl.addConstr(cum_leak[i,t] >= LEAK_TH - BIG_M_LEAK * (1 - x_this_year)) #！！！！！！

# Yearly tasks limit
for t in range(T_years):
    mdl.addConstr(quicksum(x_COPER[i,t] + x_MANT[i,t] for i in range(N)) <= L_total, name=f"Yearly_limit_t{t}")

# Annual action budget constraint
for t in range(T_years):
    yearly_action_cost = quicksum(C_COPER * x_COPER[i, t] + C_MANT * x_MANT[i, t] for i in range(N))
    mdl.addConstr(yearly_action_cost <= Total_budget, name=f"YearlyBudgetLimit[{t}]")

# objective
leak_cost = quicksum(
    (Y * alpha[i,t] * (10**5)) * K[i] * C_leak_t[t]
    for i in range(N) for t in range(T_years)
)
action_cost = quicksum(
    C_COPER * x_COPER[i,t] + C_MANT * x_MANT[i,t]
    for i in range(N) for t in range(T_years)
)
mdl.setObjective(action_cost + leak_cost, GRB.MINIMIZE)

# solve parameters
mdl.Params.MIPGap = 0.05
mdl.Params.TimeLimit = 600
mdl.Params.IntFeasTol = 1e-6
mdl.Params.NumericFocus = 2

mdl.optimize()

# output results
if mdl.status == GRB.OPTIMAL:
    print(f"\n✅ Total Cost: {mdl.objVal:,.0f}")
    
    print("\n=== Detailed Maintenance Plan and Cumulative Leak Verification ===")
    total_leak_cost = 0
    total_action_cost = 0
    
    for i in range(N):
        print(f"\n{'='*70}")
        print(f"CSEM {i+1} (Initial α={alpha_init[i]:.6f}, Volume={V[i]:.1f}m³, csemCur = {new_df['csemCur'].iloc[i]:.1f})")
        print(f"{'='*70}")
        
        csem_leak_cost = 0
        csem_action_cost = 0
        
        for t in range(T_years):
            if t == 0:
                alpha_start = alpha_init[i]
                alpha_end = alpha[i,t].X
                leak_kg = yearly_leak[i,t].X
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
            else:
                alpha_start = alpha[i,t-1].X
                alpha_end = alpha[i,t].X
                leak_kg = yearly_leak[i,t].X 
                leak_cost_t = leak_kg * C_leak_t[t]
                csem_leak_cost += leak_cost_t
                
                act_type = ""
                act_cost = 0
                if x_COPER[i,t].X > 0.5:
                    act_type = "COPER"
                    act_cost = C_COPER
                    csem_action_cost += act_cost
                elif x_MANT[i,t].X > 0.5:
                    act_type = "MANT" 
                    act_cost = C_MANT
                    csem_action_cost += act_cost
                else:
                    act_type = "No action"
                
                total_leak_cost += leak_cost_t
                if act_cost > 0:
                    total_action_cost += act_cost
                    
            alpha_reduction = (alpha_start - alpha_end) / alpha_start * 100 if alpha_start > 0 else 0
            cum_leak_val = cum_leak[i,t].X
            valid_check = "❌" if (cum_leak_val >= 2) else "✅"
            
            print(f"  Y{t+1:2d}: {act_type:<8} | "
                f"α: {alpha_start:.6f}→{alpha_end:.6f} "
                f"(-{alpha_reduction:5.1f}%) | "
                f"Leak{leak_kg:>6.3f}kg | "
                f"Cum_Leak{cum_leak_val:>6.3f}kg | {valid_check}")
        
        print(f"  CSEM{i+1} Action cost{csem_action_cost:>9,.0f} + Leak cost{csem_leak_cost:>10,.0f} = Total {csem_action_cost+csem_leak_cost:>11,.0f}")
        print(f"  Total leak: {sum(yearly_leak[i,t].X for t in range(T_years)):.1f} kg")
        print(f"  Total COPER action: {sum(x_COPER[i,t].X for t in range(T_years)):.0f}")
        print(f"  Total MANT action: {sum(x_MANT[i,t].X for t in range(T_years)):.0f}")
    
    print(f"\n{'='*70}")
    print(f"🎯 Total: action cost {total_action_cost:>12,.0f} + leak cost {total_leak_cost:>12,.0f} = total cost {mdl.objVal:>12,.0f}")
    print(f"{'='*70}")
    print(f"Total leak over entire horizon: {sum(yearly_leak[i,t].X for i in range(N) for t in range(T_years)):.1f} kg")
    print(f"Total COPER actions: {sum(x_COPER[i,t].X for i in range(N) for t in range(T_years)):.0f}")
    print(f"Total MANT actions: {sum(x_MANT[i,t].X for i in range(N) for t in range(T_years)):.0f}")

    print('h values for each CSEM over years:')
    for i in range(N):
        print(f"  CSEM {i+1}: {[h_cr for h_cr in [compute_gamma_h(Age_2023[i] + t + 1, gamma_alpha, gamma_beta) for t in range(T_years)]]}")
    print('alpha values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: { [alpha[i,t].X for t in range(T_years)] }")
    print('alpha 2023 values for each CSEM:')
    for i in range(N):
        print(f"  CSEM {i+1}: {alpha_init[i]:.6f}")


    
else:
    print(f"❌ Optimization status: {mdl.status}")


Set parameter MIPGap to value 0.05
Set parameter TimeLimit to value 600
Set parameter IntFeasTol to value 1e-06
Set parameter NumericFocus to value 2
Gurobi Optimizer version 10.0.1 build v10.0.1rc0 (mac64[arm])

CPU model: Apple M4 Pro
Thread count: 12 physical cores, 12 logical processors, using up to 12 threads

Optimize a model with 64546 rows, 23296 columns and 236768 nonzeros
Model fingerprint: 0xcfd0feb8
Variable types: 11648 continuous, 11648 integer (11648 binary)
Coefficient statistics:
  Matrix range     [3e-02, 1e+06]
  Objective range  [2e+04, 9e+07]
  Bounds range     [1e+00, 1e+00]
  RHS range        [2e-07, 1e+06]
Presolve removed 63737 rows and 22837 columns
Presolve time: 0.09s
Presolved: 809 rows, 459 columns, 3198 nonzeros
Variable types: 205 continuous, 254 integer (254 binary)
Found heuristic solution: objective 2.253016e+07

Root relaxation: objective 2.239598e+07, 180 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective B